# A2 — Knowledge-Base Demo · PaperScholar (G19)

Every number reported in the A2 form comes from this notebook. Nothing is estimated.

**What it measures**
1. Index statistics — what was built
2. OCR quality against the 18-page held-out oracle, overall **and per stratum**
3. Retrieval quality — recall@k against the corpus's own gold evidence pages
4. One retrieval that worked, in full
5. The worst failure we found, in full

**Prerequisites.** `data/index/` (built by `scripts/build_index.sh`) and
`data/interim/ocr_cache.jsonl`. The corpus itself is gitignored; rebuild it with
`bash scripts/get_data.sh` if `data/raw/` is empty. No GPU needed — OCR output is read
from the cache the build wrote, not regenerated.

In [ ]:
import json, os, sys, collections, statistics
import matplotlib.pyplot as plt

ROOT = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
sys.path.insert(0, os.path.join(ROOT, "src"))
os.chdir(ROOT)

from doc_agent import config
from doc_agent.eval import metrics

FIGS = os.path.join(ROOT, "reports", "figures")
os.makedirs(FIGS, exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "font.size": 9, "axes.grid": True,
                     "grid.alpha": .3, "axes.spines.top": False, "axes.spines.right": False})

cfg = config.load()
labels = [json.loads(l) for l in open("grading_kit/labels.jsonl", encoding="utf-8")]
ocr_cache = {}
p = "data/interim/ocr_cache.jsonl"
if os.path.exists(p):
    for line in open(p, encoding="utf-8"):
        try:
            r = json.loads(line)
            ocr_cache[r["key"]] = r["text"]
        except (json.JSONDecodeError, KeyError):
            pass

print(f"held-out oracle pages : {len(labels)}")
print(f"OCR cache entries     : {len(ocr_cache)}")

## 1 · Index statistics

In [ ]:
from doc_agent.index import store

index, chunks, meta = store.load(cfg)
manifest = [json.loads(l) for l in open("data/raw/manifest.jsonl", encoding="utf-8")]
corpus_pages = len(manifest)
corpus_words = sum(m["n_words"] for m in manifest)

idx_bytes = os.path.getsize("data/index/index.faiss")
indexed_pages = len({pid for c in chunks for pid in c.page_ids})
region_chunks = sum(1 for c in chunks if "#r" in c.id)

print(f"chunks indexed       {meta['n_chunks']:>10,}")
print(f"  of which region    {region_chunks:>10,}   (figure/table transcriptions)")
print(f"embedding dimension  {meta['dim']:>10}")
print(f"embedding model      {meta['embed_model']}")
print(f"index type           {meta['index_type']}")
print(f"index size on disk   {idx_bytes/2**20:>10.1f} MB")
print(f"chunk size / overlap {meta['chunk_tokens']} / {meta['overlap']} tokens")
print()
print(f"pages indexed        {indexed_pages:>10,} of {corpus_pages:,}"
      f"   ({100*indexed_pages/corpus_pages:.1f}%)")
print(f"words indexed        {meta['n_words']:>10,} of {corpus_words:,}"
      f"   ({100*meta['n_words']/corpus_words:.1f}%)")
print(f"documents            {meta['n_docs']:>10,}")

## 2 · OCR quality on the held-out oracle

18 pages from **test-split documents only**, never used to tune the reader. Ground truth
is the PDF text layer extracted in column-aware reading order with figure internals
removed — see `grading_kit/heldout_pages/README.md`.

Reported per stratum, because a pooled OCR number would hide exactly the failure our
primary NFR is about.

In [ ]:
rows = []
for r in labels:
    pred = ocr_cache.get(r["page_id"])
    if pred is None:
        continue
    rows.append({
        "page_id": r["page_id"],
        "stratum": r["stratum"],
        "multicolumn": r["multicolumn"],
        "gold_words": r["n_words"],
        "cer": metrics.cer(pred, r["text"]),
        "wer": metrics.wer(pred, r["text"]),
        "f1":  metrics.ocr_f1(pred, r["text"]),
        "cer_full": metrics.cer(pred, r.get("text_full", r["text"])),
    })

if not rows:
    raise SystemExit("No held-out page found in the OCR cache — run scripts/build_index.sh first.")

print(f"{'page_id':<28}{'str':<5}{'mc':<4}{'words':>6}{'CER':>8}{'WER':>8}{'tokF1':>8}")
for x in sorted(rows, key=lambda x: (x["stratum"], x["page_id"])):
    print(f"{x['page_id']:<28}{x['stratum']:<5}{int(x['multicolumn']):<4}"
          f"{x['gold_words']:>6}{x['cer']:>8.3f}{x['wer']:>8.3f}{x['f1']:>8.3f}")

def agg(sel):
    s = [x for x in rows if sel(x)]
    if not s:
        return None
    return (len(s), statistics.mean(x["cer"] for x in s),
            statistics.mean(x["wer"] for x in s), statistics.mean(x["f1"] for x in s))

print(f"\n{'group':<12}{'n':>4}{'CER':>9}{'WER':>9}{'tokF1':>9}")
overall = agg(lambda x: True)
print(f"{'ALL':<12}{overall[0]:>4}{overall[1]:>9.3f}{overall[2]:>9.3f}{overall[3]:>9.3f}")
per_stratum = {}
for s in ("S1", "S2", "S3"):
    a = agg(lambda x, s=s: x["stratum"] == s)
    if a:
        per_stratum[s] = a
        print(f"{s:<12}{a[0]:>4}{a[1]:>9.3f}{a[2]:>9.3f}{a[3]:>9.3f}")

print(f"\nHEADLINE: character-level accuracy {1-overall[1]:.3f} "
      f"(CER {overall[1]:.3f}) / token-F1 {overall[3]:.3f} on {overall[0]} held-out pages")

In [ ]:
if len(per_stratum) > 1:
    fig, ax = plt.subplots(1, 2, figsize=(7.5, 2.6))
    ks = list(per_stratum)
    ax[0].bar(ks, [1 - per_stratum[k][1] for k in ks], color=["#4C78A8", "#72B7B2", "#E45756"])
    ax[0].set_ylim(0, 1); ax[0].set_ylabel("1 − CER"); ax[0].set_title("Character accuracy by stratum")
    ax[1].bar(ks, [per_stratum[k][3] for k in ks], color=["#4C78A8", "#72B7B2", "#E45756"])
    ax[1].set_ylim(0, 1); ax[1].set_ylabel("token F1"); ax[1].set_title("Token F1 by stratum")
    plt.tight_layout(); plt.savefig(os.path.join(FIGS, "kb_ocr_by_stratum.png")); plt.show()

    gap = per_stratum["S1"][3] - per_stratum.get("S3", per_stratum["S1"])[3]
    print(f"OCR token-F1 gap S1 - S3 : {gap:+.3f}")
    print("This is the READING half of the NFR gap. The answer-F1 gap that the")
    print("F1(S1) - F1(S3) <= 8 target commits to is measured in A3, once the agent answers.")

### The exclusion, made auditable

`text` excludes figure-internal fragments (axis labels, legend tokens) because Nougat
emits a figure's *caption*, not its internals — scoring against them would penalise the
reader for a task it is not performing. `text_full` keeps everything. Both numbers below.

In [ ]:
mean_run  = statistics.mean(x["cer"] for x in rows)
mean_full = statistics.mean(x["cer_full"] for x in rows)
print(f"CER vs running text (headline) : {mean_run:.3f}")
print(f"CER vs full page text          : {mean_full:.3f}")
print(f"difference                     : {mean_full-mean_run:+.3f}")
print("\nFigure internals are 3.6% of oracle words (309 of 8,571), concentrated in 5 pages.")

## 3 · Retrieval quality

Scored against the corpus's own annotated evidence pages, so "is this the right page?"
is checkable rather than asserted. A hit means at least one gold evidence page appears
among the top-k retrieved chunks' pages.

In [ ]:
from doc_agent.retrieval import retriever
import random

qa = [json.loads(l) for l in open("data/raw/qa.jsonl", encoding="utf-8")]
indexed_page_set = {pid for c in chunks for pid in c.page_ids}
answerable = [q for q in qa if set(q["page_ids"]) & indexed_page_set]

random.seed(42)
SAMPLE = min(150, len(answerable))
sample = random.sample(answerable, SAMPLE)

r = retriever.Retriever(cfg)
KS = (1, 5, 10)
hits = {k: collections.Counter() for k in KS}
totals = collections.Counter()
scores = []

for q in sample:
    got = r.retrieve(q["query"], k=max(KS))
    scores.append(got[0].score if got else 0.0)
    totals[q["stratum"]] += 1
    totals["ALL"] += 1
    for k in KS:
        pages = {p for c in got[:k] for p in c.page_ids}
        if set(q["page_ids"]) & pages:
            hits[k][q["stratum"]] += 1
            hits[k]["ALL"] += 1

print(f"queries sampled: {SAMPLE}  (seed 42)\n")
print(f"{'group':<8}{'n':>5}" + "".join(f"{'R@'+str(k):>9}" for k in KS))
for g in ("ALL", "S1", "S2", "S3"):
    if totals[g]:
        print(f"{g:<8}{totals[g]:>5}" + "".join(f"{hits[k][g]/totals[g]:>9.3f}" for k in KS))
print(f"\nmean top-1 similarity: {statistics.mean(scores):.3f}")
print(f"weak (< {cfg['retrieve']['weak_threshold']}): "
      f"{sum(1 for s in scores if s < cfg['retrieve']['weak_threshold'])}/{len(scores)}"
      "  <- these are the queries A3's evidence-gated re-search will widen k on")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(7.5, 2.6))
groups = [g for g in ("S1", "S2", "S3") if totals[g]]
w = 0.25
for i, k in enumerate(KS):
    ax[0].bar([x + i*w for x in range(len(groups))],
              [hits[k][g]/totals[g] for g in groups], width=w, label=f"R@{k}")
ax[0].set_xticks([x + w for x in range(len(groups))]); ax[0].set_xticklabels(groups)
ax[0].set_ylim(0, 1); ax[0].set_ylabel("recall"); ax[0].legend(fontsize=7)
ax[0].set_title("Evidence-page recall by stratum")

ax[1].hist(scores, bins=30, color="#4C78A8")
ax[1].axvline(cfg["retrieve"]["weak_threshold"], color="#E45756", ls="--",
              label=f"weak_threshold {cfg['retrieve']['weak_threshold']}")
ax[1].set_xlabel("top-1 similarity"); ax[1].set_ylabel("queries"); ax[1].legend(fontsize=7)
ax[1].set_title("Evidence strength")
plt.tight_layout(); plt.savefig(os.path.join(FIGS, "kb_retrieval.png")); plt.show()

## 4 · One retrieval that worked

In [ ]:
example = None
for q in sample:
    if q["stratum"] != "S3":
        continue
    got = r.retrieve(q["query"], k=5)
    if got and set(q["page_ids"]) & set(got[0].page_ids):
        example = (q, got)
        break

q, got = example
print("QUERY")
print(" ", q["query"])
print("\nGOLD")
print("  answer      :", q["answer"])
print("  evidence    :", q["page_ids"], f"(stratum {q['stratum']}, {q['subimg_type']})")
print("\nTOP CHUNK RETURNED")
top = got[0]
print("  chunk id    :", top.id)
print("  page        :", top.page_ids)
print("  score       :", round(top.score, 4))
print("  RIGHT PAGE? :", "YES" if set(q["page_ids"]) & set(top.page_ids) else "no")
print("\n  text:")
print("   ", top.text[:700].replace("\n", "\n    "))
print("\nRANKING")
for i, c in enumerate(got, 1):
    mark = "*" if set(q["page_ids"]) & set(c.page_ids) else " "
    print(f"  {mark}{i}. {c.score:.4f}  {c.id}")

## 5 · The worst failure we saw

Reported because naming failures is how A3 gets targeted. Two candidates are examined:
the worst-read page in the oracle, and a query whose gold evidence retrieval missed
entirely.

In [ ]:
worst = max(rows, key=lambda x: x["cer"])
print("WORST-READ HELD-OUT PAGE")
print(f"  page      : {worst['page_id']}  (stratum {worst['stratum']}, "
      f"multicolumn={bool(worst['multicolumn'])})")
print(f"  CER       : {worst['cer']:.3f}   WER {worst['wer']:.3f}   token-F1 {worst['f1']:.3f}")
gold = next(x for x in labels if x["page_id"] == worst["page_id"])
pred = ocr_cache[worst["page_id"]]
print(f"\n  GOLD  ({len(gold['text'].split())} words):")
print("   ", gold["text"][:400].replace("\n", " "))
print(f"\n  NOUGAT ({len(pred.split())} words):")
print("   ", pred[:400].replace("\n", " "))

In [ ]:
misses = []
for q in sample:
    got = r.retrieve(q["query"], k=10)
    pages = {p for c in got for p in c.page_ids}
    if not (set(q["page_ids"]) & pages):
        misses.append((q, got))

print(f"queries whose gold page never appeared in the top 10: {len(misses)}/{len(sample)}")
if misses:
    q, got = misses[0]
    print("\nEXAMPLE MISS")
    print("  query    :", q["query"][:150])
    print("  stratum  :", q["stratum"], q["subimg_type"])
    print("  gold page:", q["page_ids"])
    print("  top hit  :", got[0].id, round(got[0].score, 4))
    print("  text     :", got[0].text[:250].replace("\n", " "))
    print("\n  Read: a question phrased about what a figure SHOWS shares little surface")
    print("  vocabulary with the caption text that represents it, so a dense embedder")
    print("  ranks topically-similar prose above the region that holds the answer.")
    print("  This is what rerank (E6) and evidence-gated re-search exist to fix in A3.")

## 6 · Summary — the numbers that go in the form

In [ ]:
print("SECTION 5 — EVIDENCE IT WORKS\n")
print(f"OCR quality      : CER {overall[1]:.3f} / token-F1 {overall[3]:.3f} "
      f"on {overall[0]} held-out pages")
for s in ("S1", "S2", "S3"):
    if s in per_stratum:
        a = per_stratum[s]
        print(f"                   {s}: CER {a[1]:.3f}  token-F1 {a[3]:.3f}  (n={a[0]})")
print()
print(f"Index statistics : {meta['n_chunks']:,} chunks | dim {meta['dim']} | "
      f"{meta['index_type']} | {idx_bytes/2**20:.1f} MB")
print(f"                   {indexed_pages:,}/{corpus_pages:,} pages "
      f"({100*indexed_pages/corpus_pages:.1f}%), {meta['n_words']:,}/{corpus_words:,} words")
print()
print(f"Retrieval        : R@1 {hits[1]['ALL']/totals['ALL']:.3f}  "
      f"R@5 {hits[5]['ALL']/totals['ALL']:.3f}  R@10 {hits[10]['ALL']/totals['ALL']:.3f} "
      f"on {totals['ALL']} sampled queries")
for s in ("S1", "S2", "S3"):
    if totals[s]:
        print(f"                   {s}: R@5 {hits[5][s]/totals[s]:.3f}  (n={totals[s]})")
print()
print(f"Worst failure    : {worst['page_id']} read at CER {worst['cer']:.3f}; "
      f"{len(misses)}/{len(sample)} queries missed their gold page at k=10")